In [10]:
# добавление новых фич для датасета:
import pandas as pd
import yfinance as yf
from tqdm import tqdm
import time

# 1. ЗАГРУЗКА
df = pd.read_csv(r'C:\Users\USER\Desktop\Folder\Murat\DS\Outpeer\Capstone\NewTry\trades_dataset.csv')
df['Entry Date'] = pd.to_datetime(df['Entry Date'])

# 2. ПОДГОТОВКА СПИСКА ТИКЕРОВ ДЛЯ YAHOO
unique_tickers = df['Symbol'].unique()

def get_prev_day_data(ticker_list, df_trades):
    results = {}
    print(f"Запрос рыночных данных для {len(ticker_list)} тикеров...")
    
    for ticker in tqdm(ticker_list):
        try:
            # Берем расширенное окно, чтобы точно поймать предыдущий торговый день
            data = yf.download(ticker, start="2011-12-01", end="2026-05-15", progress=False)
            if not data.empty:
                results[ticker] = data
        except:
            continue
    
    # Сопоставляем данные со сделками
    def enrich(row):
        t = row['Symbol']
        d = row['Entry Date']
        if t in results:
            t_data = results[t]
            # Ищем все дни до даты входа и берем последний доступный (предыдущий торговый день)
            prev_days = t_data[t_data.index < d]
            if not prev_days.empty:
                last_day = prev_days.iloc[-1]
                return pd.Series([last_day['Open'], last_day['Close'], last_day['Volume']])
        return pd.Series([None, None, None])

    df_trades[['Prev_Open', 'Prev_Close', 'Prev_Volume']] = df_trades.apply(enrich, axis=1)
    return df_trades

# 3. YAHOO SECTOR + ПОДГОТОВКА ДЛЯ AI
def get_sectors(df_trades):
    unique_symbols = df_trades['Symbol'].unique()
    sector_map = {}
    
    print("Получение секторов из Yahoo Finance...")
    for symbol in tqdm(unique_symbols):
        try:
            ticker_info = yf.Ticker(symbol).info
            sector_map[symbol] = ticker_info.get('sector', 'Unknown')
        except:
            sector_map[symbol] = 'Unknown'
            
    df_trades['Sector'] = df_trades['Symbol'].map(sector_map)
    
    # Выделяем тикеры для AI
    unknown_tickers = [s for s, sect in sector_map.items() if sect in ['Unknown', None, '']]
    return df_trades, unknown_tickers

# ЗАПУСК ОБОГАЩЕНИЯ
df = get_prev_day_data(unique_tickers, df)
df, tickers_for_ai = get_sectors(df)

# СОХРАНЕНИЕ
df.to_csv(r'C:\Users\USER\Desktop\Folder\Murat\DS\Outpeer\Capstone\NewTry\trades_enriched_v3.csv', index=False)

print(f"\n✅ Готово! Yahoo не смог определить сектор для {len(tickers_for_ai)} тикеров.")
print(f"tickers for AI: {tickers_for_ai}")

Запрос рыночных данных для 575 тикеров...


  0%|          | 0/575 [00:00<?, ?it/s]

$CLNT: possibly delisted; no price data found  (1d 2011-12-01 -> 2026-05-15)

1 Failed download:
['CLNT']: possibly delisted; no price data found  (1d 2011-12-01 -> 2026-05-15)
  0%|          | 1/575 [00:00<04:29,  2.13it/s]$SCEI: possibly delisted; no timezone found

1 Failed download:
['SCEI']: possibly delisted; no timezone found
  1%|          | 6/575 [00:04<06:53,  1.37it/s]$QTWW: possibly delisted; no price data found  (1d 2011-12-01 -> 2026-05-15)

1 Failed download:
['QTWW']: possibly delisted; no price data found  (1d 2011-12-01 -> 2026-05-15)
  3%|▎         | 15/575 [00:10<07:21,  1.27it/s]$LCC: possibly delisted; no price data found  (1d 2011-12-01 -> 2026-05-15)

1 Failed download:
['LCC']: possibly delisted; no price data found  (1d 2011-12-01 -> 2026-05-15)
  5%|▍         | 26/575 [00:18<06:57,  1.32it/s]$ZNGA: possibly delisted; no timezone found

1 Failed download:
['ZNGA']: possibly delisted; no timezone found
  5%|▍         | 28/575 [00:20<07:50,  1.16it/s]$GG: possib

Получение секторов из Yahoo Finance...


 68%|██████▊   | 392/575 [04:10<02:05,  1.46it/s]HTTP Error 500: <!DOCTYPE html>
<html lang="en-us">
  <head>
    <meta http-equiv="content-type" content="text/html; charset=UTF-8">
    <meta charset="utf-8">
    <title>Yahoo</title>
    <meta name="viewport" content="width=device-width,initial-scale=1,minimal-ui">
    <meta http-equiv="X-UA-Compatible" content="IE=edge,chrome=1">
    <style>
      html {
          height: 100%;
      }
      body {
          background: #fafafc url(https://s.yimg.com/nn/img/sad-panda-201402200631.png) 50% 50%;
          background-size: cover;
          height: 100%;
          text-align: center;
          font: 300 18px "helvetica neue", helvetica, verdana, tahoma, arial, sans-serif;
          margin: 0;
      }
      table {
          height: 100%;
          width: 100%;
          table-layout: fixed;
          border-collapse: collapse;
          border-spacing: 0;
          border: none;
      }
      h1 {
          font-size: 42px;
          font


✅ Готово! Yahoo не смог определить сектор для 230 тикеров.
tickers for AI: ['CLNT', 'SCEI', 'QTWW', 'UNG', 'QID', 'BOM', 'SDS', 'LCC', 'VXX', 'PCLN', 'SHLD', 'ZNGA', 'GG', 'FB', 'USO', 'SPY', 'CORN', 'ROSG', 'GLD', 'SHZ', 'UUP', 'QCOR', 'UGA', 'QQQ', 'DANG', 'XLF', 'VVUS', 'OCZ', 'MCP', 'QLD', 'RENN', 'PEIX', 'MPEL', 'DISH', 'DIA', 'FXY', 'SMH', 'RAX', 'SLV', 'EMC', 'IAU', 'MHP', 'CEF', 'IYR', 'RSOL', 'RHT', 'BBBY', 'IWM', 'SCTY', 'ARO', 'GDX', 'KORS', 'XONE', 'JCP', 'NUAN', 'TWGP', 'SINA', 'FXA', 'CELG', 'ECTE', 'NEWL', 'IBB', 'ABIO', 'TWTR', 'YHOO', 'FXC', 'AUY', 'CGA', 'JJC', 'JO', 'COG', 'SH', 'LNKD', 'WFM', 'IYT', 'FEYE', 'TBT', 'BRK.B', 'HDGE', 'EWC', 'FXE', 'PHO', 'SWN', 'SYMC', 'EWG', 'DNDN', 'X', 'SGG', 'SLW', 'DRYS', 'TAN', 'CRR', 'CREE', 'DXD', 'SWHC', 'CUBA', 'BBRY', 'JRJC', 'ARNA', 'YOKU', 'NQ', 'CHK', 'GENE', 'UCO', 'LL', 'FXP', 'EUO', 'VLTC', 'SAVE', 'VA', 'FLIR', 'BIS', 'TZA', 'GREK', 'FXI', 'JOY', 'JEC', 'XLE', 'RSX', 'SKX', 'ZFGN', 'TNA', 'POT', 'DGAZ', 'JWN', 'BIB',

In [ ]:
# выгружаем словарь из тикеров, которые не смог определить yfinance но смог gemini api
ai_sector_map = {
    # Криптовалюты (Yahoo часто падает на них с ошибкой 500)
    'BTC/USD': 'Crypto', 'ETH/USD': 'Crypto', 'LTC/USD': 'Crypto', 'XTZ/USD': 'Crypto',
    
    # Делистинги и поглощения (Technology)
    'ZNGA': 'Technology', 'TWTR': 'Technology', 'YHOO': 'Technology', 'SINA': 'Technology', 
    'LNKD': 'Technology', 'BBRY': 'Technology', 'MSFT': 'Technology', 'SCTY': 'Technology',
    'RAX': 'Technology', 'NUAN': 'Technology', 'CERN': 'Technology', 'SPLK': 'Technology',
    
    # Делистинги и поглощения (Другие сектора)
    'WFM': 'Consumer Defensive', 'KORS': 'Consumer Cyclical', 'JCP': 'Consumer Cyclical',
    'ARO': 'Consumer Cyclical', 'VA': 'Industrials', 'LCC': 'Industrials',
    'CELG': 'Healthcare', 'ARNA': 'Healthcare', 'ALXN': 'Healthcare',
    'DNDN': 'Healthcare', 'QCOR': 'Healthcare', 'CHK': 'Energy', 'COG': 'Energy',
    
    # ETF (Которые Yahoo иногда не маркирует)
    'SOXL': 'Other/ETF', 'SDOW': 'Other/ETF', 'UNG': 'Other/ETF', 'QID': 'Other/ETF',
    'MJX': 'Other/ETF', 'DWTI': 'Other/ETF'
}

In [11]:
import pandas as pd

# Загружаем обогащенный файл (v3), который создал предыдущий скрипт
df = pd.read_csv(r'C:\Users\USER\Desktop\Folder\Murat\DS\Outpeer\Capstone\NewTry\trades_enriched_v3.csv')

# 1. Применяем словарь выше для заполнения "Unknown" секторов
df['Sector'] = df['Sector'].fillna('Unknown')
for ticker, sector in ai_sector_map.items():
    df.loc[df['Symbol'] == ticker, 'Sector'] = sector

# 2. Очистка рыночных данных (если Yahoo выдал ошибки, заполняем нулями или удаляем)
# Для объема торгов и цен: если данных нет, лучше поставить 0 или среднее, 
# чтобы не удалять всю строку со сделкой.
df['Prev_Volume'] = df['Prev_Volume'].fillna(0)

# 3. Финальная проверка: считаем сколько осталось не заполненных секторов
still_unknown = df[df['Sector'] == 'Unknown']['Symbol'].unique()
print(f"Осталось неизвестных тикеров: {len(still_unknown)}")
if len(still_unknown) > 0:
    print(f"Список: {still_unknown[:10]}...") # Показываем первые 10

# 4. Сохраняем итоговый "Финальный Датасет"
df.to_csv(r'C:\Users\USER\Desktop\Folder\Murat\DS\Outpeer\Capstone\NewTry\trades_final_dataset.csv', index=False)
print("\n✅ Финальный датасет 'trades_final_golden_dataset.csv' готов к глубокому анализу!")

# import pandas as pd

# df = pd.read_csv('trades_recovered_final.csv') # или как называется твой последний файл
missing = df[df['Prev_Close'].isnull()]['Symbol'].unique()
print(list(missing))

Осталось неизвестных тикеров: 196
Список: ['CLNT' 'SCEI' 'QTWW' 'BOM' 'SDS' 'VXX' 'PCLN' 'SHLD' 'GG' 'FB']...

✅ Финальный датасет 'trades_final_golden_dataset.csv' готов к глубокому анализу!
['CLNT', 'SCEI', 'QTWW', 'LCC', 'VXX', 'PCLN', 'SHLD', 'ZNGA', 'GG', 'FB', 'ROSG', 'SHZ', 'QCOR', 'DANG', 'VVUS', 'OCZ', 'RENN', 'GNK', 'PEIX', 'MPEL', 'DISH', 'RAX', 'EMC', 'ABX', 'RSOL', 'RHT', 'SCTY', 'ARO', 'KORS', 'XONE', 'JCP', 'NUAN', 'TWGP', 'SINA', 'CELG', 'ECTE', 'NEWL', 'CCXI', 'ABIO', 'TWTR', 'YHOO', 'AUY', 'CGA', 'JJC', 'JO', 'COG', 'LNKD', 'P', 'WFM', 'FEYE', 'BRK.B', 'SNDK', 'SWN', 'SYMC', 'S', 'DNDN', 'X', 'SDRL', 'SGG', 'SLW', 'DRYS', 'Z', 'CRR', 'CREE', 'SWHC', 'CUBA', 'BBRY', 'JRJC', 'ARNA', 'YOKU', 'MOBI', 'MBLY', 'NQ', 'CHK', 'GENE', 'LL', 'VLTC', 'SAVE', 'VA', 'FLIR', 'JOY', 'JEC', 'SKX', 'ZFGN', 'POT', 'JWN', 'CA', 'VRX', 'DWTI', 'EGLE', 'SPWR', 'FH', 'ALXN', 'YY', 'MYL', 'SAEX', 'RLGY', 'EBIO', 'FIT', 'ENDP', 'SCON', 'SINO', 'ZEN', 'FL', 'APRN', 'DF', 'DCIX', 'WBA', 'ICPT',

In [12]:
import pandas as pd
import yfinance as yf
from tqdm import tqdm

# 1. Справочник маппинга (старый тикер -> актуальный или архивный)
ticker_map = {
    'FB': 'META', 'PCLN': 'BKNG', 'ANTM': 'ELV', 'TWTR': 'X', 
    'SNDK': 'WDC', 'LNKD': 'MSFT', 'YHOO': 'AABA', 'CELG': 'BMY',
    'BBRY': 'BB', 'BRK.B': 'BRK-B', 'S': 'TMUS', 'DISH': 'SATS',
    'RHT': 'IBM', 'VIAC': 'PARA', 'SQ': 'BLOCK', 'UTX': 'RTX',
    'SYMC': 'GEN', 'LCC': 'AAL', 'KORS': 'CPRI', 'RAX': 'APO',
    'WFM': 'AMZN', 'CERN': 'ORCL', 'ALXN': 'AZN', 'ZEN': 'GNE',
    'SPLK': 'CSCO', 'SWHC': 'SWBI', 'MBLY': 'MBLY', # MBLY перезапускался
    # Крипта
    'BTC/USD': 'BTC-USD', 'ETH/USD': 'ETH-USD', 
    'LTC/USD': 'LTC-USD', 'XTZ/USD': 'XTZ-USD'
}

# 2. Справочник секторов для "безнадежных" (банкроты/делистинг)
# Если цена не найдется, мы хотя бы сохраним категорию для аналитики
manual_sectors = {
    'SBNY': {'Sector': 'Financial Services', 'Industry': 'Regional Banks'},
    'JCP': {'Sector': 'Consumer Cyclical', 'Industry': 'Department Stores'},
    'DIDI': {'Sector': 'Technology', 'Industry': 'Software—Infrastructure'},
    'SCTY': {'Sector': 'Technology', 'Industry': 'Solar'},
    'ZNGA': {'Sector': 'Communication Services', 'Industry': 'Electronic Gaming'},
    'SHLD': {'Sector': 'Consumer Cyclical', 'Industry': 'Department Stores'},
    'OCZ': {'Sector': 'Technology', 'Industry': 'Computer Hardware'},
    'DRYS': {'Sector': 'Industrials', 'Industry': 'Marine Shipping'}
}

df = pd.read_csv(r'C:\Users\USER\Desktop\Folder\Murat\DS\Outpeer\Capstone\NewTry\trades_final_dataset.csv')

def final_recovery(row):
    if pd.notna(row['Prev_Close']):
        return row
    
    symbol = row['Symbol']
    # Проверяем маппинг
    target_ticker = ticker_map.get(symbol, symbol)
    
    try:
        # Пытаемся достать данные по новому тикеру или исправленному формату
        hist = yf.download(target_ticker, start=row['Entry Date'] - pd.Timedelta(days=7), 
                           end=row['Entry Date'] + pd.Timedelta(days=1), progress=False)
        if not hist.empty:
            last_day = hist[hist.index < row['Entry Date']].iloc[-1]
            row['Prev_Close'] = last_day['Close']
            # Если тикер из маппинга, пометим для истории
            if target_ticker != symbol:
                row['Notes'] = f"Recovered via {target_ticker}"
    except:
        pass
    
    # Если все еще пусто, заполняем сектор вручную (если он есть в базе)
    if pd.isna(row['Prev_Close']) and symbol in manual_sectors:
        row['Sector'] = manual_sectors[symbol]['Sector']
        row['Industry'] = manual_sectors[symbol]['Industry']
        
    return row

print("Финальная очистка данных...")
df = df.apply(final_recovery, axis=1)

# Итог
missing_after = df['Prev_Close'].isnull().sum()
print(f"После всех манипуляций осталось без цен: {missing_after}")

df.to_csv(r'C:\Users\USER\Desktop\Folder\Murat\DS\Outpeer\Capstone\NewTry\trades_final_fixed.csv', index=False)

Финальная очистка данных...
После всех манипуляций осталось без цен: 356


In [14]:
import pandas as pd

df = pd.read_csv(r'C:\Users\USER\Desktop\Folder\Murat\DS\Outpeer\Capstone\NewTry\trades_final_fixed.csv')

# Словарь секторов для тех, кого нет в Yahoo
manual_data = {
    # Технологии и Соцсети
    'FB': ('Communication Services', 'Internet Content & Information'),
    'TWTR': ('Communication Services', 'Internet Content & Information'),
    'YHOO': ('Communication Services', 'Internet Content & Information'),
    'LNKD': ('Communication Services', 'Internet Content & Information'),
    'ZNGA': ('Communication Services', 'Electronic Gaming'),
    'SINA': ('Communication Services', 'Internet Content & Information'),
    'DANG': ('Consumer Cyclical', 'Internet Retail'),
    
    # Ритейл и Потребление
    'JCP': ('Consumer Cyclical', 'Department Stores'),
    'SHLD': ('Consumer Cyclical', 'Department Stores'),
    'WFM': ('Consumer Defensive', 'Grocery Stores'),
    'KORS': ('Consumer Cyclical', 'Luxury Goods'),
    'ARO': ('Consumer Cyclical', 'Apparel Retail'),
    'APP': ('Consumer Cyclical', 'Apparel Retail'),
    
    # Энергетика и Ресурсы
    'CHK': ('Energy', 'Oil & Gas Exploration & Production'),
    'SDRL': ('Energy', 'Oil & Gas Drilling'),
    'SWN': ('Energy', 'Oil & Gas Exploration & Production'),
    'ABX': ('Basic Materials', 'Gold'),
    'GG': ('Basic Materials', 'Gold'),
    'AUY': ('Basic Materials', 'Gold'),
    'SLW': ('Basic Materials', 'Silver'),
    
    # Биотех и Медицина
    'CELG': ('Healthcare', 'Drug Manufacturers'),
    'VRX': ('Healthcare', 'Drug Manufacturers'),
    'MYL': ('Healthcare', 'Drug Manufacturers'),
    'DNDN': ('Healthcare', 'Biotechnology'),
    
    # Крипта (если вдруг не подтянулась)
    'BTC-USD': ('Financial Services', 'Digital Assets'),
    'ETH-USD': ('Financial Services', 'Digital Assets'),
}

def patch_missing_data(row):
    symbol = row['Symbol']
    
    # 1. Исправляем пустые сектора из нашего словаря
    if pd.isna(row.get('Sector')) and symbol in manual_data:
        row['Sector'] = manual_data[symbol][0]
        row['Industry'] = manual_data[symbol][1]
    
    # 2. Если цена Prev_Close все еще NaN, подставим Average Entry Price
    # Это позволит нам не терять сделку в расчетах, 
    # считая, что "вчерашняя" цена была примерно равна цене входа.
    if pd.isna(row['Prev_Close']):
        # Убираем знак $ и конвертируем в float, если нужно
        try:
            entry_price = str(row['Average Entry Price']).replace('$', '').strip()
            row['Prev_Close'] = float(entry_price)
        except:
            pass
            
    return row

df = df.apply(patch_missing_data, axis=1)

# Финальная проверка на наличие критических данных
final_missing = df['Sector'].isnull().sum()
print(f"Осталось сделок без категории: {final_missing}")

df.to_csv(r'C:\Users\USER\Desktop\Folder\Murat\DS\Outpeer\Capstone\NewTry\ready4_analyses.csv', index=False)

Осталось сделок без категории: 0


In [17]:
import pandas as pd

# 1. Загружаем текущий файл
df = pd.read_csv(r'C:\Users\USER\Desktop\Folder\Murat\DS\Outpeer\Capstone\NewTry\ready4_analyses.csv')

# 2. Списки для классификации
leveraged_tickers = [
    'SOXL', 'SOXS', 'TQQQ', 'SQQQ', 'LABU', 'LABD', 'UVXY', 'VXX', 
    'FAS', 'FAZ', 'NUGT', 'DUST', 'JNUG', 'JDST', 'TNA', 'UWT', 'DWT'
]
commodity_tickers = ['GLD', 'UNG', 'USO', 'SLW', 'PSLV', 'IAU', 'PALL', 'DBA', 'GLL']

def categorize_asset(row):
    symbol = str(row['Symbol']).upper()
    
    # Крипта
    if 'USD' in symbol or symbol in ['BTC', 'ETH', 'LTC', 'XTZ']:
        return 'Crypto'
    
    # Плечевые инструменты
    if symbol in leveraged_tickers:
        return 'Leveraged ETF'
    
    # Сырье
    if symbol in commodity_tickers:
        return 'Commodities'
    
    # Индексные фонды
    if symbol in ['SPY', 'QQQ', 'IWM', 'EEM', 'DIA']:
        return 'Index ETF'
    
    # Проверяем старый сектор
    current_sector = str(row.get('Sector', 'Unknown'))
    if current_sector not in ['Unknown', 'nan', 'Other']:
        return current_sector
        
    return 'Equity / Other'

# 3. ВОЗВРАЩАЕМ ВСЕ КОЛОНКИ (включая время и длительность)
essential_cols = [
    'Symbol', 'Long/Short', 'Entry Date', 'Average Entry Price', 
    'Exit Price', 'Exit Date', 'Profit %', 'Sector', 
    'Year', 'Month', 'Weekday', 'Trade Duration' # Вернул всё на место
]
df_final = df[[c for c in essential_cols if c in df.columns]].copy()

# 4. Применяем правила и чистим данные
df_final['Sector'] = df_final.apply(categorize_asset, axis=1)

# Чистим Profit % (убираем % и переводим в число)
if df_final['Profit %'].dtype == 'object':
    df_final['Profit %'] = df_final['Profit %'].astype(str).str.replace('%', '').astype(float)

# Даты в формат datetime
df_final['Entry Date'] = pd.to_datetime(df_final['Entry Date'])
df_final['Exit Date'] = pd.to_datetime(df_final['Exit Date'])

# 5. Сохраняем
df_final.to_csv(r'C:\Users\USER\Desktop\Folder\Murat\DS\Outpeer\Capstone\NewTry\trades_structured.csv', index=False)

print("Файл trades_structured.csv готов со всеми временными колонками!")
print(f"Колонки в новом файле: {list(df_final.columns)}")

Файл trades_structured.csv готов со всеми временными колонками!
Колонки в новом файле: ['Symbol', 'Long/Short', 'Entry Date', 'Average Entry Price', 'Exit Price', 'Exit Date', 'Profit %', 'Sector', 'Year', 'Month', 'Weekday', 'Trade Duration']


In [ ]:
# получилось несколько колонок, которые не добавляют информативности датасету, убираем их:
import pandas as pd

# Загружаем наш "мусорный" датасет
df = pd.read_csv(r'C:\Users\USER\Desktop\Folder\Murat\DS\Outpeer\Capstone\NewTry\ready4_analyses.csv')

# Список плечевых ETF (наиболее популярные в трейдинге)
leveraged_tickers = [
    'SOXL', 'SOXS', 'TQQQ', 'SQQQ', 'LABU', 'LABD', 'UVXY', 'VXX', 
    'FAS', 'FAZ', 'NUGT', 'DUST', 'JNUG', 'JDST', 'TNA', 'UWT', 'DWT'
]

# Список сырьевых тикеров
commodity_tickers = ['GLD', 'UNG', 'USO', 'SLW', 'PSLV', 'IAU', 'PALL', 'DBA']

def categorize_asset(row):
    symbol = str(row['Symbol']).upper()
    
    # 1. Проверка на Крипту
    if 'USD' in symbol or symbol in ['BTC', 'ETH', 'LTC', 'XTZ']:
        return 'Crypto'
    
    # 2. Проверка на плечевые инструменты
    if symbol in leveraged_tickers:
        return 'Leveraged ETF'
    
    # 3. Проверка на сырье (Commodities)
    if symbol in commodity_tickers:
        return 'Commodities'
    
    # 4. Проверка на обычные ETF (не плечевые)
    if symbol in ['SPY', 'QQQ', 'IWM', 'EEM', 'DIA']:
        return 'Index ETF'
    
    # 5. Если сектор уже определен как акция (Technology, Healthcare и т.д.)
    current_sector = str(row.get('Sector', 'Unknown'))
    if current_sector not in ['Unknown', 'nan', 'Other']:
        return current_sector
        
    return 'Equity / Other'

# Очищаем датасет
essential_cols = [
    'Symbol', 'Long/Short', 'Entry Date', 'Average Entry Price', 
    'Exit Price', 'Exit Date', 'Profit %', 'Sector', 'Year'
]
df_final = df[[c for c in essential_cols if c in df.columns]].copy()

# Применяем новую категоризацию в колонку Sector
df_final['Sector'] = df_final.apply(categorize_asset, axis=1)

# Чистим Profit % (важно для расчетов)
if df_final['Profit %'].dtype == 'object':
    df_final['Profit %'] = df_final['Profit %'].str.replace('%', '').astype(float)

# Сортировка по времени
df_final['Entry Date'] = pd.to_datetime(df_final['Entry Date'])
df_final = df_final.sort_values('Entry Date')

# Сохраняем очищенный результат для дальнейшего анализа
df_final.to_csv(r'C:\Users\USER\Desktop\Folder\Murat\DS\Outpeer\Capstone\NewTry\trades_structured.csv', index=False)

print("Данные структурированы по новым правилам!")
print(df_final['Sector'].value_counts()) # Посмотрим распределение+

Данные структурированы по новым правилам!
Sector
Equity / Other            560
Technology                416
Consumer Cyclical         277
Communication Services    143
Leveraged ETF             140
Financial Services        134
Industrials                99
Healthcare                 92
Energy                     74
Consumer Defensive         53
Commodities                43
Basic Materials            41
Other/ETF                  29
Index ETF                  25
Utilities                  15
Crypto                     15
Real Estate                12
Name: count, dtype: int64


NameError: name 'df' is not defined